In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import os

# ============ Config ============
experiment_name = "strong_aug"
log_dir = f"runs/{experiment_name}"
save_path = f"runs/{experiment_name}/best.pth"
num_epochs = 20
batch_size = 64
learning_rate = 0.001
# ================================

# 1. Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. TensorBoard SummaryWriter
writer = SummaryWriter(log_dir)

# 3. Transform
transform = transforms.Compose([
    transforms.RandomResizedCrop(32, scale=(0.8, 1.2)),         # Random scale & crop
    transforms.RandomHorizontalFlip(0.9),                          # Flip
    transforms.ColorJitter(0.8, 0.8, 0.8, 0.4),                 # Brightness, Contrast, etc.
    transforms.RandomGrayscale(p=0.9),                          # 20% chance grayscale
    transforms.RandomAffine(degrees=45, translate=(0.3, 0.3), scale=(0.5, 1.5)),  # Rotate & shift
    transforms.RandomPerspective(distortion_scale=1, p=3),  # Simulate 3D distortions
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),   # Blur simulating motion/sensor
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), 
                         (0.5, 0.5, 0.5))
])

# 4. Dataset & DataLoader
train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

# 5. Model (fix warning: use weights=None)
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 10)
model.to(device)

# 6. Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 7. Evaluation function
def evaluate():
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

# 8. Training function
def train_one_epoch(epoch):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

    avg_loss = running_loss / len(train_loader)
    train_acc = correct / total
    val_acc = evaluate()

    # Log to TensorBoard
    writer.add_scalar("Loss/train", avg_loss, epoch)
    writer.add_scalar("Accuracy/train", train_acc, epoch)
    writer.add_scalar("Accuracy/val", val_acc, epoch)

    print(f"Epoch [{epoch+1}] Train Loss: {avg_loss:.4f}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")
    return val_acc

# 9. Training loop with best model saving
best_val_acc = 0.0
for epoch in range(num_epochs):
    val_acc = train_one_epoch(epoch)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), save_path)
        print(f"✅ New best model saved with val acc: {best_val_acc:.4f}")

writer.close()

Epoch [1] Train Loss: 2.3553, Train Acc: 0.1062, Val Acc: 0.1213
✅ New best model saved with val acc: 0.1213
Epoch [2] Train Loss: 2.3350, Train Acc: 0.1149, Val Acc: 0.0982
Epoch [3] Train Loss: 2.3322, Train Acc: 0.1123, Val Acc: 0.1041
Epoch [4] Train Loss: 2.3299, Train Acc: 0.1112, Val Acc: 0.1232
✅ New best model saved with val acc: 0.1232
Epoch [5] Train Loss: 2.3228, Train Acc: 0.1132, Val Acc: 0.1084
Epoch [6] Train Loss: 2.3187, Train Acc: 0.1181, Val Acc: 0.1181
Epoch [7] Train Loss: 2.3151, Train Acc: 0.1187, Val Acc: 0.1197
Epoch [8] Train Loss: 2.3049, Train Acc: 0.1250, Val Acc: 0.1207
Epoch [9] Train Loss: 2.3007, Train Acc: 0.1287, Val Acc: 0.1263
✅ New best model saved with val acc: 0.1263
Epoch [10] Train Loss: 2.2956, Train Acc: 0.1263, Val Acc: 0.1251
Epoch [11] Train Loss: 2.2904, Train Acc: 0.1295, Val Acc: 0.1313
✅ New best model saved with val acc: 0.1313
Epoch [12] Train Loss: 2.2862, Train Acc: 0.1315, Val Acc: 0.1245
Epoch [13] Train Loss: 2.2828, Train Acc: